# Text-to-SQL: Fine-tuning for Natural Language to SQL Generation

Fine-tunes a small LLM (CodeLlama-7B or similar) with LoRA on Text-to-SQL datasets.

**Run all cells top to bottom.**

In [1]:
# ============================================================
# CELL 1: Install dependencies
# ============================================================
!pip install -q datasets transformers[torch] accelerate peft bitsandbytes trl sqlparse rouge-score

In [2]:
# ============================================================
# CELL 2: Imports
# ============================================================
import os, json, torch, sqlparse
import numpy as np
from datasets import load_dataset, Dataset, concatenate_datasets
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

PyTorch: 2.10.0+cu128
CUDA: True
GPU: Tesla T4


In [3]:
# ============================================================
# CELL 3: Configuration
# ============================================================
MODEL_NAME = 'defog/sqlcoder-7b-2'  # Specialized Text-to-SQL model
OUTPUT_DIR = './text2sql-finetuned'

# Training params
MAX_SAMPLES = 10000
MAX_EVAL = 1000
MAX_SEQ_LENGTH = 1024
BATCH_SIZE = 4
GRADIENT_ACCUMULATION = 4
LEARNING_RATE = 2e-4
NUM_EPOCHS = 3
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# Our database schema (what the model needs to learn)
SCHEMA = '''CREATE TABLE stock_prices (
    id SERIAL PRIMARY KEY,
    ticker VARCHAR(10) NOT NULL,
    date DATE NOT NULL,
    open FLOAT, high FLOAT, low FLOAT, close FLOAT,
    volume BIGINT,
    UNIQUE(ticker, date)
);

CREATE TABLE financial_metrics (
    id SERIAL PRIMARY KEY,
    ticker VARCHAR(10),
    company VARCHAR(100),
    metric_name VARCHAR(50) NOT NULL,
    value FLOAT,
    period VARCHAR(10),
    report_date DATE,
    UNIQUE(ticker, metric_name, period)
);

CREATE TABLE macro_indicators (
    id SERIAL PRIMARY KEY,
    country VARCHAR(50),
    indicator_name VARCHAR(150) NOT NULL,
    value FLOAT,
    unit VARCHAR(30),
    date DATE,
    source VARCHAR(100)
);'''

print(f'Model: {MODEL_NAME}')
print(f'Max samples: {MAX_SAMPLES}')

Model: defog/sqlcoder-7b-2
Max samples: 10000


In [4]:
# ============================================================
# CELL 4: Load datasets
# ============================================================
# WikiSQL + Spider + custom financial examples
print('Loading datasets...')

# 1. Spider dataset (complex SQL)
spider = load_dataset('xlangai/spider', split='train')
spider_eval = load_dataset('xlangai/spider', split='validation')
print(f'  Spider: {len(spider)} train / {len(spider_eval)} eval')

# 2. sql-create-context (question + schema -> SQL)
sql_context = load_dataset('b-mc2/sql-create-context', split='train')
print(f'  sql-create-context: {len(sql_context)}')

print('Datasets loaded.')

Loading datasets...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


  Spider: 7000 train / 1034 eval
  sql-create-context: 78577
Datasets loaded.


In [5]:
# ============================================================
# CELL 5: Create custom financial training examples
# ============================================================
# These teach the model our specific schema
custom_examples = [
    {'question': "What is NVIDIA's stock price?", 'sql': "SELECT ticker, date, close FROM stock_prices WHERE ticker = 'NVDA' ORDER BY date DESC LIMIT 1;"},
    {'question': "Show me Apple's price history for the last month", 'sql': "SELECT date, open, high, low, close, volume FROM stock_prices WHERE ticker = 'AAPL' AND date >= CURRENT_DATE - INTERVAL '30 days' ORDER BY date;"},
    {'question': "Compare P/E ratios of AAPL and MSFT", 'sql': "SELECT ticker, value FROM financial_metrics WHERE ticker IN ('AAPL', 'MSFT') AND metric_name = 'trailing_p_e';"},
    {'question': "Which stock had the highest volume this week?", 'sql': "SELECT ticker, date, volume FROM stock_prices WHERE date >= CURRENT_DATE - INTERVAL '7 days' ORDER BY volume DESC LIMIT 1;"},
    {'question': "What is the gold price today?", 'sql': "SELECT indicator_name, value, date FROM macro_indicators WHERE indicator_name LIKE '%Gold%' ORDER BY date DESC LIMIT 1;"},
    {'question': "Show me the VIX trend over 3 months", 'sql': "SELECT date, value FROM macro_indicators WHERE indicator_name LIKE '%VIX%' AND date >= CURRENT_DATE - INTERVAL '90 days' ORDER BY date;"},
    {'question': "What is the EUR/USD exchange rate?", 'sql': "SELECT indicator_name, value, date FROM macro_indicators WHERE indicator_name LIKE '%EUR/USD%' ORDER BY date DESC LIMIT 1;"},
    {'question': "Which stocks have dividend yield above 2%?", 'sql': "SELECT ticker, value FROM financial_metrics WHERE metric_name = 'dividend_yield' AND value > 0.02 ORDER BY value DESC;"},
    {'question': "What is Tesla's market cap?", 'sql': "SELECT ticker, value FROM financial_metrics WHERE ticker = 'TSLA' AND metric_name = 'market_cap';"},
    {'question': "Show me all stocks with P/E below 20", 'sql': "SELECT ticker, value FROM financial_metrics WHERE metric_name = 'trailing_p_e' AND value < 20 ORDER BY value;"},
    {'question': "What was the S&P 500 last week?", 'sql': "SELECT date, value FROM macro_indicators WHERE indicator_name LIKE '%S&P 500%' AND date >= CURRENT_DATE - INTERVAL '7 days' ORDER BY date;"},
    {'question': "Compare Brent and WTI oil prices", 'sql': "SELECT indicator_name, value, date FROM macro_indicators WHERE indicator_name LIKE '%Oil%' ORDER BY date DESC LIMIT 10;"},
    {'question': "Top 5 stocks by market cap", 'sql': "SELECT ticker, value FROM financial_metrics WHERE metric_name = 'market_cap' ORDER BY value DESC LIMIT 5;"},
    {'question': "NVIDIA revenue vs Microsoft revenue", 'sql': "SELECT ticker, value FROM financial_metrics WHERE ticker IN ('NVDA', 'MSFT') AND metric_name = 'revenue';"},
    {'question': "Average closing price of AAPL last 30 days", 'sql': "SELECT AVG(close) as avg_close FROM stock_prices WHERE ticker = 'AAPL' AND date >= CURRENT_DATE - INTERVAL '30 days';"},
    {'question': "Quel est le cours de NVIDIA?", 'sql': "SELECT ticker, date, close FROM stock_prices WHERE ticker = 'NVDA' ORDER BY date DESC LIMIT 1;"},
    {'question': "Montrez-moi le prix de l'or", 'sql': "SELECT indicator_name, value, date FROM macro_indicators WHERE indicator_name LIKE '%Gold%' ORDER BY date DESC LIMIT 1;"},
    {'question': "Comparer le P/E de Apple et Google", 'sql': "SELECT ticker, value FROM financial_metrics WHERE ticker IN ('AAPL', 'GOOGL') AND metric_name = 'trailing_p_e';"},
]

print(f'Custom financial examples: {len(custom_examples)}')

Custom financial examples: 18


In [6]:
# ============================================================
# CELL 6: Format training data
# ============================================================
PROMPT_TEMPLATE = '''### Task
Generate a SQL query to answer the following question.

### Database Schema
{schema}

### Question
{question}

### SQL
{sql}'''

def format_spider(example):
    return {'text': PROMPT_TEMPLATE.format(
        schema=example.get('create_table', SCHEMA),
        question=example['question'],
        sql=example['query']
    )}

def format_sql_context(example):
    return {'text': PROMPT_TEMPLATE.format(
        schema=example.get('context', SCHEMA),
        question=example['question'],
        sql=example['answer']
    )}

def format_custom(example):
    return {'text': PROMPT_TEMPLATE.format(
        schema=SCHEMA,
        question=example['question'],
        sql=example['sql']
    )}

# Format all datasets
spider_formatted = spider.map(format_spider, remove_columns=spider.column_names)
sql_context_formatted = sql_context.map(format_sql_context, remove_columns=sql_context.column_names)
custom_ds = Dataset.from_list(custom_examples).map(format_custom, remove_columns=['question', 'sql'])

# Repeat custom examples 50x to emphasize our schema
custom_repeated = concatenate_datasets([custom_ds] * 50)

# Combine and subsample
all_train = concatenate_datasets([
    spider_formatted.shuffle(seed=42).select(range(min(4000, len(spider_formatted)))),
    sql_context_formatted.shuffle(seed=42).select(range(min(5000, len(sql_context_formatted)))),
    custom_repeated,
]).shuffle(seed=42)

# Eval set from Spider validation
eval_ds = spider_eval.map(format_spider, remove_columns=spider_eval.column_names)
if len(eval_ds) > MAX_EVAL:
    eval_ds = eval_ds.shuffle(seed=42).select(range(MAX_EVAL))

print(f'Training samples: {len(all_train)}')
print(f'Eval samples: {len(eval_ds)}')
print(f'\nSample:\n{all_train[0]["text"][:300]}')

Map:   0%|          | 0/18 [00:00<?, ? examples/s]

Training samples: 9900
Eval samples: 1000

Sample:
### Task
Generate a SQL query to answer the following question.

### Database Schema
CREATE TABLE stock_prices (
    id SERIAL PRIMARY KEY,
    ticker VARCHAR(10) NOT NULL,
    date DATE NOT NULL,
    open FLOAT, high FLOAT, low FLOAT, close FLOAT,
    volume BIGINT,
    UNIQUE(ticker, date)
);

CRE


In [7]:
# ============================================================
# CELL 7: Load model with 4-bit quantization (QLoRA)
# ============================================================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)


print(f'Loading {MODEL_NAME} in 4-bit...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.float16,
)
model = prepare_model_for_kbit_training(model)

print(f'Model loaded. Parameters: {model.num_parameters() / 1e9:.1f}B (4-bit quantized)')

Loading defog/sqlcoder-7b-2 in 4-bit...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Model loaded. Parameters: 6.7B (4-bit quantized)


In [8]:
# ============================================================
# CELL 8: Configure LoRA adapters
# ============================================================
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)

model = get_peft_model(model, lora_config)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable parameters: {trainable/1e6:.1f}M / {total/1e6:.1f}M ({100*trainable/total:.2f}%)')

Trainable parameters: 40.0M / 3540.5M (1.13%)


In [9]:

import trl
print(trl.__version__)
help(SFTTrainer.__init__)


1.4.0
Help on function __init__ in module trl.trainer.sft_trainer:

__init__(self, model: 'str | PreTrainedModel | PeftModel', args: trl.trainer.sft_config.SFTConfig | transformers.training_args.TrainingArguments | None = None, data_collator: collections.abc.Callable[[list[typing.Any]], dict[str, typing.Any]] | None = None, train_dataset: datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset | None = None, eval_dataset: datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset | dict[str, datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset] | None = None, processing_class: transformers.tokenization_utils_base.PreTrainedTokenizerBase | transformers.processing_utils.ProcessorMixin | None = None, compute_loss_func: collections.abc.Callable | None = None, compute_metrics: collections.abc.Callable[[transformers.trainer_utils.EvalPrediction], dict] | None = None, callbacks: list[transformers.trainer_callback.TrainerCallback] | None

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    warmup_ratio=0.05,
    fp16=False,
    bf16=False,
    max_grad_norm=0,
    logging_steps=25,
    eval_strategy='steps',
    eval_steps=200,
    save_strategy='steps',
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to='tensorboard',
    optim='paged_adamw_8bit',
)


# Truncate dataset texts to max length via tokenizer
tokenizer.model_max_length = MAX_SEQ_LENGTH

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=all_train,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
)

print('Starting training...')
train_result = trainer.train()

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f'\nDone. Loss: {train_result.training_loss:.4f}')
print(f'Model saved to {OUTPUT_DIR}')


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Starting training...


Step,Training Loss,Validation Loss
200,0.288078,0.182327


In [ ]:
# ============================================================
# CELL 10: Evaluation - Exact Match + Execution Accuracy
# ============================================================
from transformers import pipeline

print('Loading fine-tuned model for evaluation...')
sql_pipe = pipeline(
    'text-generation',
    model=OUTPUT_DIR,
    tokenizer=OUTPUT_DIR,
    max_new_tokens=256,
    temperature=0.0,
    do_sample=False,
    device_map='auto',
)

# Test on our financial schema
test_questions = [
    "What is NVIDIA's current stock price?",
    "Show me Apple's P/E ratio",
    "Which stock had the highest volume today?",
    "What is the gold price?",
    "Compare market cap of AAPL and MSFT",
    "Show me the VIX over the last week",
    "What is the EUR/USD rate?",
    "Top 3 stocks by dividend yield",
    "Average NVDA closing price last 30 days",
    "Quel est le cours de Tesla?",
]

INFERENCE_TEMPLATE = '''### Task
Generate a SQL query to answer the following question.

### Database Schema
{schema}

### Question
{question}

### SQL
'''

print(f"{'Question':<45} {'Generated SQL'}")
print('=' * 100)

results = []
for q in test_questions:
    prompt = INFERENCE_TEMPLATE.format(schema=SCHEMA, question=q)
    output = sql_pipe(prompt)[0]['generated_text']
    # Extract SQL after the prompt
    sql = output[len(prompt):].strip().split('\n')[0].strip()
    results.append({'question': q, 'sql': sql})
    print(f"{q:<45} {sql[:55]}")

print('\n' + '=' * 100)

In [ ]:
# ============================================================
# CELL 11: Quantitative evaluation metrics
# ============================================================
def normalize_sql(sql):
    """Normalize SQL for comparison."""
    sql = sql.strip().rstrip(';').lower()
    sql = ' '.join(sql.split())  # collapse whitespace
    return sqlparse.format(sql, strip_comments=True, reindent=False)

def is_valid_sql(sql):
    """Check if SQL is syntactically valid."""
    try:
        parsed = sqlparse.parse(sql)
        return len(parsed) > 0 and parsed[0].get_type() in ('SELECT', 'UNKNOWN')
    except:
        return False

# Evaluate on custom examples (we know the ground truth)
correct = 0
valid = 0
total = len(custom_examples)

print(f"{'Question':<40} {'Valid':<6} {'Match'}")
print('-' * 70)

for ex in custom_examples:
    prompt = INFERENCE_TEMPLATE.format(schema=SCHEMA, question=ex['question'])
    output = sql_pipe(prompt)[0]['generated_text']
    predicted = output[len(prompt):].strip().split('\n')[0].strip()
    
    is_valid = is_valid_sql(predicted)
    is_match = normalize_sql(predicted) == normalize_sql(ex['sql'])
    
    if is_valid: valid += 1
    if is_match: correct += 1
    
    print(f"{ex['question'][:38]:<40} {'Y' if is_valid else 'N':<6} {'Y' if is_match else 'N'}")

print('-' * 70)
print(f'\nResults on {total} financial queries:')
print(f'  SQL Validity Rate: {valid}/{total} ({100*valid/total:.1f}%)')
print(f'  Exact Match Rate:  {correct}/{total} ({100*correct/total:.1f}%)')

In [ ]:
# ============================================================
# CELL 12: Visualize results
# ============================================================
import matplotlib.pyplot as plt

metrics = ['SQL Validity', 'Exact Match']
scores = [100*valid/total, 100*correct/total]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(metrics, scores, color=['#10b981', '#2d4a36'], width=0.5)
ax.set_ylabel('Score (%)')
ax.set_title('Text-to-SQL Model Evaluation on Financial Queries')
ax.set_ylim(0, 105)

for bar, val in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            f'{val:.1f}%', ha='center', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('text2sql_evaluation.png', dpi=150)
plt.show()
print('Chart saved.')

In [ ]:
# ============================================================
# CELL 13: Save summary
# ============================================================
summary = {
    'model_base': MODEL_NAME,
    'method': 'QLoRA (4-bit quantization + LoRA adapters)',
    'datasets': ['xlangai/spider', 'b-mc2/sql-create-context', 'custom_financial'],
    'training': {
        'epochs': NUM_EPOCHS,
        'batch_size': BATCH_SIZE * GRADIENT_ACCUMULATION,
        'learning_rate': LEARNING_RATE,
        'lora_r': LORA_R,
        'train_samples': len(all_train),
    },
    'results': {
        'sql_validity_rate': round(100*valid/total, 1),
        'exact_match_rate': round(100*correct/total, 1),
    },
}

with open(os.path.join(OUTPUT_DIR, 'training_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)

print('Summary saved.')
print(f'\nTo use this model:')
print(f'  from transformers import pipeline')
print(f'  sql_gen = pipeline("text-generation", model="{OUTPUT_DIR}")')
print(f'  result = sql_gen(prompt)[0]["generated_text"]')